# Sentiment Analysis using NLP Pipeline & ML Models
**Internship Task 2: February 2026**
**Developed by:** Shankar Reddy

In [1]:
# Install and download all dependencies in one go
!pip install pandas numpy scikit-learn nltk
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to C:\Users\Mopur Shankar
[nltk_data]     Reddy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Mopur Shankar
[nltk_data]     Reddy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Mopur Shankar
[nltk_data]     Reddy\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Mopur Shankar
[nltk_data]     Reddy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset (Ensure IMDB Dataset.csv is uploaded)
df = pd.read_csv('IMDB Dataset.csv')

# Use a subset for faster training (20% of data)
df = df.sample(10000, random_state=42)
print("Dataset loaded. Shape:", df.shape)
print(df['sentiment'].value_counts())

Dataset loaded. Shape: (10000, 2)
sentiment
positive    5039
negative    4961
Name: count, dtype: int64


In [7]:
def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Removing HTML tags
    text = re.sub(r'<br />', ' ', text)
    # 3. Removing Special Characters & Punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 4. Tokenization & Stopword Removal
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [w for w in words if w not in stop_words]
    
    # 5. Lemmatization
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return " ".join(words)

# Apply preprocessing
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing Complete. Sample:")
print(df[['review', 'cleaned_review']].head(2))

Preprocessing Complete. Sample:
                                                  review  \
33553  I really liked this Summerslam due to the look...   
9427   Not many television shows appeal to quite as m...   

                                          cleaned_review  
33553  really liked summerslam due look arena curtain...  
9427   many television show appeal quite many differe...  


In [8]:
X = df['cleaned_review']
y = df['sentiment'].replace({'positive': 1, 'negative': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Bag of Words (BoW)
cv = CountVectorizer(max_features=5000)
X_train_bow = cv.fit_transform(X_train)
X_test_bow = cv.transform(X_test)

# 2. TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Vectorization complete.")

C:\Users\Mopur Shankar Reddy\AppData\Local\Temp\ipykernel_30108\2570436784.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df['sentiment'].replace({'positive': 1, 'negative': 0})


Vectorization complete.


In [9]:
def train_and_evaluate(X_tr, X_te, y_tr, y_te, method_name):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Naive Bayes": MultinomialNB(),
        "Decision Tree": DecisionTreeClassifier(max_depth=10)
    }
    
    print(f"\n--- Results for {method_name} ---")
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
        print(f"{name} Accuracy: {acc:.4f}")

# Compare both vectorizers
train_and_evaluate(X_train_bow, X_test_bow, y_train, y_test, "Bag of Words")
train_and_evaluate(X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")


--- Results for Bag of Words ---
Logistic Regression Accuracy: 0.8415
Naive Bayes Accuracy: 0.8435
Decision Tree Accuracy: 0.7235

--- Results for TF-IDF ---
Logistic Regression Accuracy: 0.8720
Naive Bayes Accuracy: 0.8495
Decision Tree Accuracy: 0.7295


### Final Insights:
1. **Best Model:** Logistic Regression and Naive Bayes performed significantly better than the Decision Tree. This is because high-dimensional text data is better handled by statistical models than simple tree splits.
2. **Best Vectorization:** TF-IDF showed slightly better performance than BoW because it reduces the weight of common words and highlights unique keywords.
3. **Preprocessing:** Removing HTML tags and Lemmatization were essential steps to ensure the model was learning from clean, meaningful text.